# Graph-based multi-hop retrieval — one variable at a time

This notebook runs parameter sweeps on Colab, **one variable at a time**, comparing results
across different values while keeping everything else at the `config/base.yaml` default.

Each section below targets one config knob. Run a section’s cells in order; results land in
`results/<experiment_id>/` and are compared in the final section.

**Before running:** *Runtime → Change runtime type → T4 GPU*

Budget roughly 20–30 minutes for the first experiment (graph build), then 5–10 minutes for
each additional value that reuses those graphs.

## 1. Clone the repo

In [ ]:
REF = "main"  # branch for iteration, or a commit SHA to pin a sweep exactly
REPO_URL = "https://github.com/hadasy-tau/graphs_project.git"

import os

REPO = "/content/graphs_project"

if os.path.isdir(REPO):
    !cd {REPO} && git fetch --all --quiet && git checkout {REF} && git pull --ff-only || true
else:
    !git clone {REPO_URL} {REPO} && cd {REPO} && git checkout {REF}

os.chdir(REPO)  # every later cell, shell command included, runs from the repo root
!git log --oneline -1

## 2. Install dependencies

Two to three minutes. `pcst-fast` compiles from C++ source here — fine on Linux, and stage 3
cannot run without it.

In [ ]:
# requirements.txt pins the en_core_web_lg 3.7.1 wheel. That pin drags spaCy back to 3.7.x
# and numpy below 2.0 with it, a downgrade the already-running kernel only picks up after a
# restart. So install everything else from the file and let spaCy fetch the model build that
# matches whatever version it resolved - same NER model, no downgrade, no restart.
lines = [l for l in open("requirements.txt").read().splitlines() if "en-core-web" not in l]
with open("/tmp/requirements-colab.txt", "w") as f:
    f.write("\n".join(lines) + "\n")

!pip install -q -r /tmp/requirements-colab.txt
!python -m spacy download en_core_web_lg

In [ ]:
# Sanity check: fail here rather than 10 minutes into a stage.
import spacy
import torch

print("torch     :", torch.__version__)
print("GPU       :", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "NONE - Runtime > Change runtime type > T4 GPU")

spacy.load("en_core_web_lg")
print("spaCy NER : en_core_web_lg OK")

try:
    import pcst_fast  # noqa: F401
    print("pcst_fast : OK (the real Goemans-Williamson solver)")
except ImportError:
    print("pcst_fast : MISSING - stage 3 will fail. Re-run `pip install pcst-fast` and read its error.")

In [ ]:
# The config every experiment starts from.
print(open("config/base.yaml").read())

## 3. Shared setup

Stage 01 (download, clean, embed, NER) was already run and its outputs are saved under
`data/`. The sweeps below start from stage 02 (graph build) onward.

Run this cell once before any sweep section below.

In [ ]:
import json
import shlex
import sys

sys.path.insert(0, "scripts")
from experiment import GRAPH_KEYS, apply_overrides, expand_sweep, fingerprint, load_config

SWEEP_MODE = "axis"        # one key at a time: 3 values = 3 experiments (+ baseline)
INCLUDE_BASELINE = True    # run an all-defaults arm called "baseline" once per section
STAGES = "02,03,04"        # stage 01 already done; data/ is ready
EXTRA_ARGS = ""            # e.g. "--force" to rerun a section from scratch

base_cfg = load_config("config/base.yaml")


def preview(sweep):
    """Show which experiment folders the sweep will create, then return the arms list."""
    arms = expand_sweep(sweep, mode=SWEEP_MODE, include_baseline=INCLUDE_BASELINE)
    print(f"{len(arms)} experiment(s):\n")
    graph_ids = []
    for exp_id, overrides in arms:
        merged = apply_overrides(base_cfg, overrides, allow_new_keys=False)
        graph_ids.append(fingerprint(merged, GRAPH_KEYS))
        print(f"  results/{exp_id:<30} graphs g-{graph_ids[-1][:6]}  {overrides}")
    print(f"\n{len(set(graph_ids))} graph build(s) for {len(arms)} experiment(s)")
    return arms

## 4. Sweeps — run one section at a time

Each section below tests one config knob. Run the two cells in a section:
1. **Preview cell** — sets `SWEEP` and shows which experiment folders will be created.
2. **Run cell** — executes the sweep; streams output live.

Results accumulate in `results/`; the comparison section at the bottom works across all of them.
Arms that share a graph fingerprint build those graphs once — so a prior section’s graphs are
reused automatically.

---
### mutual_knn_k

Controls how many mutual k-NN edges each node keeps in the semantic **and** metadata graphs.
`null` in `base.yaml` means “match the entity graph’s average degree” (≈ 17 on this dataset).
Pinning it lets you trade recall against density independently of the entity graph.

In [ ]:
# Default: null → auto ≈17.  Sparser (10) and denser (25) variants.
SWEEP = {"mutual_knn_k": [10, 17, 25]}
arms = preview(SWEEP)

In [ ]:
args = (f"--sweep-json {shlex.quote(json.dumps(SWEEP))} --sweep-mode {SWEEP_MODE} "
        f"{'--include-baseline ' if INCLUDE_BASELINE else ''}--stages {STAGES} {EXTRA_ARGS}")
!python -u scripts/experiment.py {args}
print("\nexit code:", _exit_code, "(0 = every experiment finished every stage)")

---
### entity_graph.min_shared_entities

Minimum number of shared named entities required to draw an edge in the entity graph.
Higher values produce a sparser, higher-precision graph; lower values add more recall at the
cost of noisy edges.

In [ ]:
# Default: 2.  Looser (1) and stricter (4) variants.
SWEEP = {"entity_graph.min_shared_entities": [1, 2, 4]}
arms = preview(SWEEP)

In [ ]:
args = (f"--sweep-json {shlex.quote(json.dumps(SWEEP))} --sweep-mode {SWEEP_MODE} "
        f"{'--include-baseline ' if INCLUDE_BASELINE else ''}--stages {STAGES} {EXTRA_ARGS}")
!python -u scripts/experiment.py {args}
print("\nexit code:", _exit_code, "(0 = every experiment finished every stage)")

---
### metadata_graph.fields

Which document-level metadata fields are used to build the metadata graph.
More fields add more cross-document edges; fewer fields make the graph sparser and more precise.

In [ ]:
# Default: [title, author].  Title-only (sparser) and +source (denser) variants.
SWEEP = {"metadata_graph.fields": [
    ["title"],
    ["title", "author"],
    ["title", "author", "source"],
]}
arms = preview(SWEEP)

In [ ]:
args = (f"--sweep-json {shlex.quote(json.dumps(SWEEP))} --sweep-mode {SWEEP_MODE} "
        f"{'--include-baseline ' if INCLUDE_BASELINE else ''}--stages {STAGES} {EXTRA_ARGS}")
!python -u scripts/experiment.py {args}
print("\nexit code:", _exit_code, "(0 = every experiment finished every stage)")

---
### pcst.topk

Number of top-k prize-collecting Steiner tree nodes to retrieve per query.
This is a retrieval-only knob: it reuses graphs already built by earlier sections.

In [ ]:
# Default: 6.  Fewer (3) and more (9) nodes per query.
SWEEP = {"pcst.topk": [3, 6, 9]}
arms = preview(SWEEP)

In [ ]:
args = (f"--sweep-json {shlex.quote(json.dumps(SWEEP))} --sweep-mode {SWEEP_MODE} "
        f"{'--include-baseline ' if INCLUDE_BASELINE else ''}--stages {STAGES} {EXTRA_ARGS}")
!python -u scripts/experiment.py {args}
print("\nexit code:", _exit_code, "(0 = every experiment finished every stage)")

---
### pcst.cost_e

Edge cost in the PCST solver — how expensive it is to include an edge in the Steiner tree.
Higher values produce smaller, sparser trees; lower values allow more edges.

In [ ]:
# Default: 0.5.  Cheaper (0.25) and more expensive (0.75) edge costs.
SWEEP = {"pcst.cost_e": [0.25, 0.5, 0.75]}
arms = preview(SWEEP)

In [ ]:
args = (f"--sweep-json {shlex.quote(json.dumps(SWEEP))} --sweep-mode {SWEEP_MODE} "
        f"{'--include-baseline ' if INCLUDE_BASELINE else ''}--stages {STAGES} {EXTRA_ARGS}")
!python -u scripts/experiment.py {args}
print("\nexit code:", _exit_code, "(0 = every experiment finished every stage)")

---
### pcst.combined_cost_e

Edge cost used in the combined multi-graph PCST run.
Analogous to `cost_e` but applies when edges from all three graphs compete in a single solve.

In [ ]:
# Default: 0.25.  Cheaper (0.1) and more expensive (0.5) variants.
SWEEP = {"pcst.combined_cost_e": [0.1, 0.25, 0.5]}
arms = preview(SWEEP)

In [ ]:
args = (f"--sweep-json {shlex.quote(json.dumps(SWEEP))} --sweep-mode {SWEEP_MODE} "
        f"{'--include-baseline ' if INCLUDE_BASELINE else ''}--stages {STAGES} {EXTRA_ARGS}")
!python -u scripts/experiment.py {args}
print("\nexit code:", _exit_code, "(0 = every experiment finished every stage)")

---
## 5. Compare the experiments

Every `results/*/metrics/summary_table.csv`, stacked with the experiment id and its overrides,
then pivoted so each condition is a row and each experiment a column. Also saved to
`results/comparison_all_experiments.csv`.

In [ ]:
from pathlib import Path

import pandas as pd

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)

frames = []
for summary in sorted(Path("results").glob("*/metrics/summary_table.csv")):
    exp_dir = summary.parents[1]
    manifest_path = exp_dir / "manifest.json"
    manifest = json.loads(manifest_path.read_text(encoding="utf-8")) if manifest_path.exists() else {}
    df = pd.read_csv(summary)
    df.insert(0, "experiment_id", exp_dir.name)
    df.insert(1, "overrides", json.dumps(manifest.get("overrides", {}), sort_keys=True))
    df.insert(2, "run_status", manifest.get("status", "unknown"))
    frames.append(df)

comparison = pd.concat(frames, ignore_index=True)
comparison.to_csv("results/comparison_all_experiments.csv", index=False)

print("=== every condition, every experiment ===")
display(comparison.sort_values(["condition", "experiment_id"]))

print("=== F1 ===")
display(comparison.pivot(index="condition", columns="experiment_id", values="f1"))

print("=== evidence recall ===")
display(comparison.pivot(index="condition", columns="experiment_id", values="evidence_recall"))

In [ ]:
# Did the knob actually change the graph? For a mutual_knn_k sweep the semantic and
# metadata edge counts are the direct read-out; for metadata_graph.fields, the metadata
# row alone. If these are identical across arms the override never reached the builders.
stats = []
for path in sorted(Path("results").glob("*/metrics/graph_stats.csv")):
    df = pd.read_csv(path)
    df.insert(0, "experiment_id", path.parents[1].name)
    stats.append(df)

graph_stats = pd.concat(stats, ignore_index=True)
display(graph_stats.pivot(index="name", columns="experiment_id", values="n_edges"))
display(graph_stats.pivot(index="name", columns="experiment_id", values="oracle_connectivity"))

## 6. One experiment in detail

Its config, what each stage cost, and every metrics table it produced.

In [ ]:
EXPERIMENT_ID = "baseline"  # or any folder name, e.g. "mutual_knn_k17"

exp_dir = Path("results") / EXPERIMENT_ID
manifest = json.loads((exp_dir / "manifest.json").read_text(encoding="utf-8"))

print(f"=== {EXPERIMENT_ID}: {manifest['status']} ===")
print(f"overrides {json.dumps(manifest['overrides'], sort_keys=True)}")
print(f"git       {manifest['git']['sha'][:10]} on {manifest['git']['branch']}"
      f"{' (dirty)' if manifest['git']['dirty'] else ''}")
print(f"graphs    {manifest['paths']['graphs']}\n")
display(pd.DataFrame(manifest["stage_runs"])[["script", "status", "duration_s"]])

print(f"\n=== {exp_dir / 'config.yaml'} ===")
print((exp_dir / "config.yaml").read_text(encoding="utf-8"))

for csv in sorted((exp_dir / "metrics").glob("*.csv")):
    print(f"\n=== {csv} ===")
    display(pd.read_csv(csv))

print("\nlogs:", ", ".join(p.name for p in sorted((exp_dir / "logs").glob("*.log"))))

## 7. Download everything

All of `results/` — every experiment’s metrics, retrieval output, logs and the config that
produced them. The caches under `data/` are left behind: hundreds of MB, and regenerated from
the model checkpoints.

In [ ]:
import os
import shutil

BUNDLE = "/content/graphs_project_experiments"
shutil.rmtree(BUNDLE, ignore_errors=True)
shutil.copytree("results", BUNDLE)

zip_path = shutil.make_archive(BUNDLE, "zip", BUNDLE)
print(f"{zip_path}  ({os.path.getsize(zip_path) / 1e6:.1f} MB)")

from google.colab import files
files.download(zip_path)

## Keeping the caches between sessions

`data/` dies with the Colab runtime, so the next session re-downloads and re-embeds everything.
To keep it, mount Drive and point the caches at it **before** running any sweep — the
fingerprinted folder names mean nothing goes stale:

```python
from google.colab import drive
drive.mount("/content/drive")
os.environ["GRAPHS_PROJECT_DATA_ROOT"] = "/content/drive/MyDrive/graphs_project/data"
```

`results/` stays inside the clone; the download cell above is how you keep it. Both are
gitignored, so the only committed record of a sweep is the cells above — keep them committed,
and pin `REF` to a commit SHA if you want a manifest’s git SHA to still mean something later.